# Layer 3 — Quality Tests

Validates all four Layer 3 pipeline steps:
- `3.1_dependency_graph.py` → `rca_graph_results.csv`
- `3.2_causal_inference.py` → `rca_causal_results.csv`
- `3.3_external_drivers.py` → `rca_results.csv`
- `3.4_rca_assembly.py`    → `rca_assembly.csv`

| # | Test | What it checks |
|---|------|----------------|
| 1 | Intermediate file shapes | Four output files with correct (rows, cols) |
| 2 | Traversal depth distribution | Tier 1 depth breakdown; Tier 2/3 no-traversal count |
| 3 | Traversal stop reason breakdown | tier_2_3_no_traversal / no_anomalous_child / leaf_node_reached |
| 4 | Tier 2/3 linkage to Tier 1 | All 93 Tier 2/3 anomalies trace back to a Tier 1 KPI |
| 5 | CausalImpact coverage and significance | 14 of 15 HIGH ran; 6 significant effects |
| 6 | DoWhy coverage and refutation | 26 anomalies; 7 unique pairs; all 26 refutations passed |
| 7 | Root cause confidence quality | 35 available; 13 of 15 HIGH above 0.70 |
| 8 | External driver type distribution | 8 externally driven; 6 suppressed |
| 9 | Suppression integrity | No HIGH or UP anomaly ever suppressed |
| 10 | Black Friday spot-check | is_externally_driven=False, escalation_suppressed=False |
| 11 | Layer 4 priority flag distribution | ESCALATE 15 / INVESTIGATE 86 / MONITOR 74 / SUPPRESSED 6 |
| 12 | Assembly merge integrity + SQLite parity | actual/expected non-null; all 4 SQLite tables have 181 rows |

In [1]:
import pandas as pd
import sqlite3
import os
from pathlib import Path

# Resolve project root regardless of launch directory
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

DATA = PROJECT_ROOT / 'data'

graph  = pd.read_csv(DATA / 'rca_graph_results.csv')
causal = pd.read_csv(DATA / 'rca_causal_results.csv')
rca    = pd.read_csv(DATA / 'rca_results.csv')
asm    = pd.read_csv(DATA / 'rca_assembly.csv')

print(f'Project root : {PROJECT_ROOT}')
print(f'rca_graph_results  : {graph.shape}')
print(f'rca_causal_results : {causal.shape}')
print(f'rca_results        : {rca.shape}')
print(f'rca_assembly       : {asm.shape}')

Project root : C:\Users\annes\OneDrive\Adidas Office One Drive - Personal Folders\Upskilling\2026\KPI Anomaly Detection Agent
rca_graph_results  : (181, 18)
rca_causal_results : (181, 40)
rca_results        : (181, 52)
rca_assembly       : (181, 45)


---
## Test 1 — Intermediate File Shapes
Expected:
```
PASS  rca_graph_results.csv        (181, 18)
PASS  rca_causal_results.csv       (181, 40)
PASS  rca_results.csv              (181, 52)
PASS  rca_assembly.csv             (181, 45)
```

In [2]:
expected = {
    'rca_graph_results.csv':  (181, 18),
    'rca_causal_results.csv': (181, 40),
    'rca_results.csv':        (181, 52),
    'rca_assembly.csv':       (181, 45),
}
for fname, (exp_r, exp_c) in expected.items():
    df = pd.read_csv(DATA / fname)
    assert df.shape == (exp_r, exp_c), \
        f'{fname}: expected ({exp_r}, {exp_c}), got {df.shape}'
    print(f'PASS  {fname:<28}  {df.shape}')

PASS  rca_graph_results.csv         (181, 18)
PASS  rca_causal_results.csv        (181, 40)
PASS  rca_results.csv               (181, 52)
PASS  rca_assembly.csv              (181, 45)


---
## Test 2 — Step 3.1 Traversal Depth Distribution
Expected:
```
PASS  Tier 1 traversals -- depth=0: 62  depth=1: 22  depth=2: 4
PASS  Tier 2/3 (no traversal): 93  (19 Tier 2 + 74 Tier 3)
```

In [3]:
tier1 = graph[graph['tier'] == 1]
depth = tier1['graph_depth_reached'].value_counts().sort_index()

assert depth.get(0, 0) == 62, f'Expected 62 depth-0 Tier 1, got {depth.get(0,0)}'
assert depth.get(1, 0) == 22, f'Expected 22 depth-1 Tier 1, got {depth.get(1,0)}'
assert depth.get(2, 0) == 4,  f'Expected 4  depth-2 Tier 1, got {depth.get(2,0)}'

tier23_n = (graph['traversal_stopped'] == 'tier_2_3_no_traversal').sum()
assert tier23_n == 93, f'Expected 93 Tier 2/3 no-traversal, got {tier23_n}'

print(f'PASS  Tier 1 traversals -- depth=0: {depth.get(0,0)}  depth=1: {depth.get(1,0)}  depth=2: {depth.get(2,0)}')
print(f'PASS  Tier 2/3 (no traversal): {tier23_n}  (19 Tier 2 + 74 Tier 3)')

PASS  Tier 1 traversals -- depth=0: 62  depth=1: 22  depth=2: 4
PASS  Tier 2/3 (no traversal): 93  (19 Tier 2 + 74 Tier 3)


---
## Test 3 — Step 3.1 Traversal Stop Reason Breakdown
Expected:
```
PASS  tier_2_3_no_traversal :   93  (Tier 2/3 are already the driver)
PASS  no_anomalous_child     :   73  (no child above |z| >= 1.5)
PASS  leaf_node_reached      :   15  (traversal reached a leaf node)
PASS  total                  : 181
```

In [4]:
stops = graph['traversal_stopped'].value_counts()

assert stops.get('tier_2_3_no_traversal', 0) == 93
assert stops.get('no_anomalous_child',    0) == 73
assert stops.get('leaf_node_reached',     0) == 15

print(f"PASS  tier_2_3_no_traversal :  {stops.get('tier_2_3_no_traversal', 0):>3}  (Tier 2/3 are already the driver)")
print(f"PASS  no_anomalous_child     :  {stops.get('no_anomalous_child', 0):>3}  (no child above |z| >= 1.5)")
print(f"PASS  leaf_node_reached      :  {stops.get('leaf_node_reached', 0):>3}  (traversal reached a leaf node)")
print(f'PASS  total                  : {len(graph)}')

PASS  tier_2_3_no_traversal :   93  (Tier 2/3 are already the driver)
PASS  no_anomalous_child     :   73  (no child above |z| >= 1.5)
PASS  leaf_node_reached      :   15  (traversal reached a leaf node)
PASS  total                  : 181


---
## Test 4 — Step 3.1 Tier 2/3 Linkage to Tier 1
Expected:
```
PASS  All 93 Tier 2/3 anomalies linked to >= 1 Tier 1 KPI via dependency graph
```

In [5]:
tier23 = graph[graph['tier'].isin([2, 3])]
linked = (tier23['affected_tier1_kpis'].fillna('').str.strip() != '').sum()

assert linked == len(tier23), \
    f'Expected all {len(tier23)} Tier 2/3 rows linked, got {linked}'

print(f'PASS  All {linked} Tier 2/3 anomalies linked to >= 1 Tier 1 KPI via dependency graph')

PASS  All 93 Tier 2/3 anomalies linked to >= 1 Tier 1 KPI via dependency graph


---
## Test 5 — Step 3.2 CausalImpact Coverage and Significance
Expected:
```
PASS  CausalImpact ran:       14 / 15 HIGH anomalies
PASS  1 skipped (pre_period < 30 rows): ANO-20240111-ROAS
PASS  Significant effects:    6 / 14  (CI excludes zero)
```

In [6]:
ci_ran = int(causal['ci_ran'].sum())
ci_sig = int(causal.loc[causal['ci_ran'] == True, 'ci_effect_significant'].sum())
high_n = int((causal['severity'] == 'HIGH').sum())

assert ci_ran == 14
assert ci_sig == 6
assert high_n == 15

skipped = causal[(causal['severity'] == 'HIGH') & (causal['ci_ran'] == False)]
assert skipped.iloc[0]['anomaly_id'] == 'ANO-20240111-ROAS'

print(f'PASS  CausalImpact ran:       {ci_ran} / {high_n} HIGH anomalies')
print(f'PASS  1 skipped (pre_period < 30 rows): ANO-20240111-ROAS')
print(f'PASS  Significant effects:    {ci_sig} / {ci_ran}  (CI excludes zero)')

PASS  CausalImpact ran:       14 / 15 HIGH anomalies
PASS  1 skipped (pre_period < 30 rows): ANO-20240111-ROAS
PASS  Significant effects:    6 / 14  (CI excludes zero)


---
## Test 6 — Step 3.2 DoWhy Coverage and Refutation
Expected:
```
PASS  DoWhy ran:              26 anomalies  (HIGH + MEDIUM with distinct driver)
PASS  Refutation passed:      26 / 26  (all estimates robust)
PASS  Unique (driver -> outcome) pairs: 7
```

In [7]:
dw_ran    = int(causal['dw_ran'].sum())
dw_ref_ok = int(causal.loc[causal['dw_ran'] == True, 'dw_refutation_passed'].sum())
pairs     = causal.loc[causal['dw_ran'] == True, ['dw_treatment', 'dw_outcome']].drop_duplicates()

assert dw_ran == 26
assert dw_ref_ok == 26
assert len(pairs) == 7

print(f'PASS  DoWhy ran:              {dw_ran} anomalies  (HIGH + MEDIUM with distinct driver)')
print(f'PASS  Refutation passed:      {dw_ref_ok} / {dw_ran}  (all estimates robust)')
print(f'PASS  Unique (driver -> outcome) pairs: {len(pairs)}')
print()
print('Pairs estimated:')
for _, p in pairs.iterrows():
    row = causal[(causal['dw_treatment'] == p['dw_treatment']) & (causal['dw_outcome'] == p['dw_outcome'])].iloc[0]
    print(f"  {p['dw_treatment']:<25} -> {p['dw_outcome']:<25}  ATE={row['dw_ate_coeff']:.6f}  p={row['dw_p_value']:.4f}")

PASS  DoWhy ran:              26 anomalies  (HIGH + MEDIUM with distinct driver)
PASS  Refutation passed:      26 / 26  (all estimates robust)
PASS  Unique (driver -> outcome) pairs: 7

Pairs estimated:
  n_orders                  -> total_revenue_usd          ATE=57.385187  p=0.0309
  sessions                  -> total_revenue_usd          ATE=0.030446  p=0.0000
  avg_order_value_usd       -> total_revenue_usd          ATE=202.564455  p=0.0033
  sessions                  -> n_orders                   ATE=0.000828  p=0.0000
  avg_discount_pct          -> n_orders                   ATE=972.935110  p=0.0000
  total_clicks              -> avg_roas                   ATE=0.000623  p=0.0000
  avg_discount_pct          -> conversion_rate            ATE=0.221135  p=0.0000


---
## Test 7 — Step 3.2 Root Cause Confidence Quality
Expected:
```
PASS  root_cause_confidence available: 35 / 181 anomalies
PASS  All available values in [0, 1]
PASS  HIGH anomalies > 0.70: 13 / 15  (mean = 0.86)
```

In [8]:
rcc     = causal['root_cause_confidence'].dropna()
high_rc = causal.loc[causal['severity'] == 'HIGH', 'root_cause_confidence'].dropna()
above07 = int((high_rc > 0.70).sum())
mean_h  = round(float(high_rc.mean()), 3)

assert len(rcc) == 35
assert rcc.between(0, 1).all()
assert above07 >= 10

print(f'PASS  root_cause_confidence available: {len(rcc)} / {len(causal)} anomalies')
print(f'PASS  All available values in [0, 1]')
print(f'PASS  HIGH anomalies > 0.70: {above07} / {len(high_rc)}  (mean = {mean_h})')
print()
print('HIGH anomaly confidence detail (descending):')
high_detail = causal[causal['severity'] == 'HIGH'][['anomaly_id', 'kpi', 'root_cause_confidence']].sort_values('root_cause_confidence', ascending=False)
print(high_detail.to_string(index=False))

PASS  root_cause_confidence available: 35 / 181 anomalies
PASS  All available values in [0, 1]
PASS  HIGH anomalies > 0.70: 13 / 15  (mean = 0.86)

HIGH anomaly confidence detail (descending):
       anomaly_id               kpi  root_cause_confidence
 ANO-20241129-ORD          n_orders                 1.0000
 ANO-20251128-ORD          n_orders                 1.0000
 ANO-20241129-REV total_revenue_usd                 0.9600
 ANO-20240820-ORD          n_orders                 0.9600
 ANO-20251128-REV total_revenue_usd                 0.9600
 ANO-20240820-REV total_revenue_usd                 0.9600
 ANO-20241202-REV total_revenue_usd                 0.9192
ANO-20240111-ROAS          avg_roas                 0.9000
ANO-20250525-ROAS          avg_roas                 0.8946
ANO-20250924-ROAS          avg_roas                 0.8487
 ANO-20241202-ORD          n_orders                 0.8385
ANO-20241129-ROAS          avg_roas                 0.7584
ANO-20240131-ROAS          avg_roas     

---
## Test 8 — Step 3.3 External Driver Type Distribution
Expected:
```
PASS  Externally driven:          8 / 181
PASS  competitive_pressure:       6  (avg_roas DOWN, marketing_pressure > 0.30, actionability=0.45)
PASS  consumer_sentiment_decline: 2  (return_rate UP, sentiment < -0.10, actionability=0.90)
PASS  Escalation suppressed:      6  (competitive_pressure cases only)
```

In [9]:
ext_n  = int(rca['is_externally_driven'].sum())
sup_n  = int(rca['escalation_suppressed'].sum())
comp_n = int(rca['external_driver_type'].str.contains('competitive_pressure', na=False).sum())
sent_n = int(rca['external_driver_type'].str.contains('consumer_sentiment_decline', na=False).sum())

assert ext_n  == 8
assert sup_n  == 6
assert comp_n == 6
assert sent_n == 2

print(f'PASS  Externally driven:          {ext_n} / {len(rca)}')
print(f'PASS  competitive_pressure:       {comp_n}  (avg_roas DOWN, marketing_pressure > 0.30, actionability=0.45)')
print(f'PASS  consumer_sentiment_decline: {sent_n}  (return_rate UP, sentiment < -0.10, actionability=0.90)')
print(f'PASS  Escalation suppressed:      {sup_n}  (competitive_pressure cases only)')
print()
print('Suppressed anomalies:')
sup_cols = ['anomaly_id', 'date', 'kpi', 'direction', 'deviation_pct',
            'snap_marketing_pressure', 'actionability_score', 'suppression_reason']
print(rca[rca['escalation_suppressed']][sup_cols].to_string(index=False))

PASS  Externally driven:          8 / 181
PASS  competitive_pressure:       6  (avg_roas DOWN, marketing_pressure > 0.30, actionability=0.45)
PASS  consumer_sentiment_decline: 2  (return_rate UP, sentiment < -0.10, actionability=0.90)
PASS  Escalation suppressed:      6  (competitive_pressure cases only)

Suppressed anomalies:
       anomaly_id       date      kpi direction  deviation_pct  snap_marketing_pressure  actionability_score                                    suppression_reason
ANO-20240618-ROAS 2024-06-18 avg_roas      DOWN         -35.21                  0.36144                 0.45 Suppressed: competitive_pressure (actionability=0.45)
ANO-20241214-ROAS 2024-12-14 avg_roas      DOWN         -20.14                  0.38398                 0.45 Suppressed: competitive_pressure (actionability=0.45)
ANO-20241216-ROAS 2024-12-16 avg_roas      DOWN         -53.82                  0.35268                 0.45 Suppressed: competitive_pressure (actionability=0.45)
ANO-20250128-ROAS 2

---
## Test 9 — Step 3.3 Suppression Integrity
Expected:
```
PASS  HIGH anomalies suppressed:   0  (none -- HIGH always escalates)
PASS  UP anomalies suppressed:     0  (positive spikes never suppressed)
PASS  All 6 suppressed rows are DOWN direction and MEDIUM severity
```

In [10]:
high_sup = rca[(rca['severity'] == 'HIGH') & rca['escalation_suppressed']]
up_sup   = rca[(rca['direction'] == 'UP')  & rca['escalation_suppressed']]
sup      = rca[rca['escalation_suppressed']]

assert len(high_sup) == 0
assert len(up_sup)   == 0
assert (sup['direction'] == 'DOWN').all()
assert (sup['severity']  == 'MEDIUM').all()

print(f'PASS  HIGH anomalies suppressed:   {len(high_sup)}  (none -- HIGH always escalates)')
print(f'PASS  UP anomalies suppressed:     {len(up_sup)}  (positive spikes never suppressed)')
print(f'PASS  All {len(sup)} suppressed rows are DOWN direction and MEDIUM severity')

PASS  HIGH anomalies suppressed:   0  (none -- HIGH always escalates)


PASS  UP anomalies suppressed:     0  (positive spikes never suppressed)
PASS  All 6 suppressed rows are DOWN direction and MEDIUM severity


---
## Test 10 — Step 3.3 Black Friday Spot-Check
Expected:
```
PASS  2024-11-29  total_revenue_usd  direction=UP  dev=+223.83%
PASS  is_externally_driven=False   escalation_suppressed=False
PASS  actionability_score=1.0  root_cause_confidence=0.96
```

In [11]:
bf = rca[(rca['date'] == '2024-11-29') & (rca['kpi'] == 'total_revenue_usd')].iloc[0]

assert not bf['is_externally_driven']
assert not bf['escalation_suppressed']
assert bf['actionability_score'] == 1.0
assert bf['direction'] == 'UP'

print(f"PASS  2024-11-29  total_revenue_usd  direction={bf['direction']}  dev={bf['deviation_pct']:+.2f}%")
print(f"PASS  is_externally_driven={bf['is_externally_driven']}   escalation_suppressed={bf['escalation_suppressed']}")
print(f"PASS  actionability_score={bf['actionability_score']}  root_cause_confidence={bf['root_cause_confidence']}")
print()
print('Full rca_narrative:')
print(bf['rca_narrative'])

PASS  2024-11-29  total_revenue_usd  direction=UP  dev=+223.83%
PASS  is_externally_driven=False   escalation_suppressed=False
PASS  actionability_score=1.0  root_cause_confidence=0.96

Full rca_narrative:
[HIGH] total_revenue_usd moved UP +223.8% on 2024-11-29. Chain: total_revenue_usd -> avg_order_value_usd. Suspected driver: avg_order_value_usd. Causal confidence: 96%. No external suppression. Fully actionable.


---
## Test 11 — Step 3.4 Layer 4 Priority Flag Distribution
Expected:
```
PASS  ESCALATE    :  15  (HIGH severity -- all three methods agree)
PASS  INVESTIGATE :  86  (MEDIUM -- actionable, not suppressed)
PASS  MONITOR     :  74  (LOW / Tier 3 -- daily digest only)
PASS  SUPPRESSED  :   6  (externally driven competitive pressure)
PASS  Total       : 181
```

In [12]:
flags = asm['layer4_priority_flag'].value_counts()

assert flags.get('ESCALATE',    0) == 15
assert flags.get('INVESTIGATE', 0) == 86
assert flags.get('MONITOR',     0) == 74
assert flags.get('SUPPRESSED',  0) == 6
assert flags.sum() == len(asm)

print(f"PASS  ESCALATE    : {flags.get('ESCALATE',    0):>3}  (HIGH severity -- all three methods agree)")
print(f"PASS  INVESTIGATE : {flags.get('INVESTIGATE', 0):>3}  (MEDIUM -- actionable, not suppressed)")
print(f"PASS  MONITOR     : {flags.get('MONITOR',     0):>3}  (LOW / Tier 3 -- daily digest only)")
print(f"PASS  SUPPRESSED  : {flags.get('SUPPRESSED',  0):>3}  (externally driven competitive pressure)")
print(f'PASS  Total       : {len(asm)}')
print()
print('RCA completeness score (0 = graph only, 3 = all three methods):')
print(asm['rca_completeness_score'].value_counts().sort_index().to_string())

PASS  ESCALATE    :  15  (HIGH severity -- all three methods agree)
PASS  INVESTIGATE :  86  (MEDIUM -- actionable, not suppressed)
PASS  MONITOR     :  74  (LOW / Tier 3 -- daily digest only)
PASS  SUPPRESSED  :   6  (externally driven competitive pressure)
PASS  Total       : 181

RCA completeness score (0 = graph only, 3 = all three methods):
rca_completeness_score
0    146
1      9
2     21
3      5


---
## Test 12 — Step 3.4 Assembly Merge Integrity + SQLite Parity
Expected:
```
PASS  actual_value non-null:   181 / 181
PASS  expected_value non-null: 181 / 181
PASS  All 181 anomaly IDs unique
PASS  rca_graph_results          181 rows
PASS  rca_causal_results         181 rows
PASS  rca_results                181 rows
PASS  rca_assembly               181 rows
```

In [13]:
null_actual   = int(asm['actual_value'].isna().sum())
null_expected = int(asm['expected_value'].isna().sum())

assert null_actual   == 0
assert null_expected == 0
assert asm['anomaly_id'].nunique() == 181

print(f'PASS  actual_value non-null:   {len(asm) - null_actual} / {len(asm)}')
print(f'PASS  expected_value non-null: {len(asm) - null_expected} / {len(asm)}')
print(f'PASS  All 181 anomaly IDs unique')

conn = sqlite3.connect(DATA / 'kpi_anomaly_detection.db')
for table in ['rca_graph_results', 'rca_causal_results', 'rca_results', 'rca_assembly']:
    n = conn.execute(f'SELECT COUNT(*) FROM [{table}]').fetchone()[0]
    assert n == 181, f'{table}: expected 181 rows, got {n}'
    print(f'PASS  {table:<25}  {n} rows')
conn.close()

PASS  actual_value non-null:   181 / 181
PASS  expected_value non-null: 181 / 181
PASS  All 181 anomaly IDs unique
PASS  rca_graph_results          181 rows
PASS  rca_causal_results         181 rows
PASS  rca_results                181 rows
PASS  rca_assembly               181 rows


---
## Summary

In [14]:
tests = [
    ('Test 1',  'Intermediate file shapes'),
    ('Test 2',  'Traversal depth distribution'),
    ('Test 3',  'Traversal stop reason breakdown'),
    ('Test 4',  'Tier 2/3 linkage to Tier 1'),
    ('Test 5',  'CausalImpact coverage and significance'),
    ('Test 6',  'DoWhy coverage and refutation'),
    ('Test 7',  'Root cause confidence quality'),
    ('Test 8',  'External driver type distribution'),
    ('Test 9',  'Suppression integrity'),
    ('Test 10', 'Black Friday spot-check'),
    ('Test 11', 'Layer 4 priority flag distribution'),
    ('Test 12', 'Assembly merge integrity + SQLite parity'),
]

print('=' * 58)
print('Layer 3 Quality Test Results')
print('=' * 58)
for num, name in tests:
    print(f'  PASS  {num:<8} {name}')
print('=' * 58)
print(f'  All 12 tests passed -- Layer 3 output is valid.')
print(f'  rca_assembly.csv is ready for Layer 4.')
print('=' * 58)
print()
print('Layer 3 output files:')
print(f'  rca_graph_results.csv  : {graph.shape}')
print(f'  rca_causal_results.csv : {causal.shape}')
print(f'  rca_results.csv        : {rca.shape}')
print(f'  rca_assembly.csv       : {asm.shape}  <-- Layer 4 handoff')

Layer 3 Quality Test Results
  PASS  Test 1   Intermediate file shapes
  PASS  Test 2   Traversal depth distribution
  PASS  Test 3   Traversal stop reason breakdown
  PASS  Test 4   Tier 2/3 linkage to Tier 1
  PASS  Test 5   CausalImpact coverage and significance
  PASS  Test 6   DoWhy coverage and refutation
  PASS  Test 7   Root cause confidence quality
  PASS  Test 8   External driver type distribution
  PASS  Test 9   Suppression integrity
  PASS  Test 10  Black Friday spot-check
  PASS  Test 11  Layer 4 priority flag distribution
  PASS  Test 12  Assembly merge integrity + SQLite parity
  All 12 tests passed -- Layer 3 output is valid.
  rca_assembly.csv is ready for Layer 4.

Layer 3 output files:
  rca_graph_results.csv  : (181, 18)
  rca_causal_results.csv : (181, 40)
  rca_results.csv        : (181, 52)
  rca_assembly.csv       : (181, 45)  <-- Layer 4 handoff
